## Script to fine-tue LLMs based on data generated by llm vs. llm

In [1]:
import re
import os
import math
import json
from copy import deepcopy
from typing import List, Dict, Tuple, Optional
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
# QLoRA / quantization imports
from transformers import BitsAndBytesConfig, EarlyStoppingCallback, EvalPrediction
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
import wandb
from huggingface_hub import login
import random
from sklearn.model_selection import train_test_split
import numpy as np
import shutil

/project/train_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open('/project/hf_api_login_key.txt', 'r') as f:
    hf_api = f.read()
login(token=hf_api)

In [3]:
# ------------------------------------------------------------------
# put these at the VERY TOP of your script, before importing HF libs
# ------------------------------------------------------------------
os.environ["HF_HOME"] = "/data/huggingface"
os.environ["HUGGINGFACE_HUB_CACHE"] = "/data/huggingface"
os.environ["WANDB_DISABLED"] = "false"   # flip to "false" if you actually want wandb

OUTPUT_DIR = "/project/testing_finetuning"  # your requested path
os.makedirs(OUTPUT_DIR, exist_ok=True)




In [4]:
# ---------------------------
# 0) Load data
# ---------------------------
with open('./referencegame_data_llms_DS_v0.json', 'r') as f:
    data_v0 = json.load(f)

with open('./referencegame_data_llms_DS_v1.json', 'r') as f:
    data_v1 = json.load(f)
"""   
with open('./referencegame_data_DS_v2.json', 'r') as f:
    data_v2 = json.load(f)
"""
data = data_v0 + data_v1 #+ data_v2

for i, datapoint in enumerate(data):
    datapoint["id"] = i

In [5]:
# -----------------------------------------
# 1) Dataset + collator
# -----------------------------------------
class BanditDataset(Dataset):
    def __init__(self, rows: List[dict], tokenizer: AutoTokenizer, max_len: int = 4096, include_answer=True):
        self.rows = rows
        self.tok = tokenizer
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.max_len = max_len
        self.include_answer = include_answer

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        if r["role"] == "gen":
            prompt, action = r["prompt_speaker"], r["utterance"]
        else:
            prompt, action = r["prompt_listener"], r["guess"]

        # target for comprehension evaluation
        target_text = r["target_text"] 
        
        enc_p = self.tok(prompt, add_special_tokens=False)
        
        if self.include_answer:
            
            action_with_eos = action + self.tok.eos_token
            enc_a = self.tok(action_with_eos, add_special_tokens=False)

            input_ids = enc_p["input_ids"] + enc_a["input_ids"]
            labels = [-100] * len(enc_p["input_ids"]) + enc_a["input_ids"]
            
        else: # no included answer for the dev set 
            input_ids = enc_p["input_ids"]
            labels = [-100] * len(enc_p["input_ids"])
            
        attn = [1] * len(input_ids)

        if len(input_ids) > self.max_len:
            overflow = len(input_ids) - self.max_len
            input_ids = input_ids[overflow:]
            labels = labels[overflow:]
            attn = attn[overflow:]

        logp_behavior = r.get("logp_behavior", None)
        has_behav = float((logp_behavior is not None) and (r["reward"] == -1))

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attn, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "reward": torch.tensor(float(r["reward"]), dtype=torch.float),
            "logp_behavior": torch.tensor(0.0 if logp_behavior is None else float(logp_behavior), dtype=torch.float),
            "has_behav": torch.tensor(has_behav, dtype=torch.float),
            # keep gold answer for evaluation
            "target_text": target_text,  
        }

    
class PadCollator:
    def __init__(self, pad_id: int, padding_side:str):
        self.pad_id = pad_id
        self.padding_side = padding_side


    def __call__(self, batch: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
        max_len = max(x["input_ids"].shape[0] for x in batch)
        for x in batch:
            pad = max_len - x["input_ids"].shape[0]
            if pad > 0:
                x["input_ids"] = torch.nn.functional.pad(x["input_ids"], (pad, 0), value=self.pad_id)
                x["attention_mask"] = torch.nn.functional.pad(x["attention_mask"], (pad, 0), value=0)
                x["labels"] = torch.nn.functional.pad(x["labels"], (pad, 0), value=-100)
        out = {k: torch.stack([x[k] for x in batch]) for k in ["input_ids", "attention_mask", "labels", "reward", "logp_behavior", "has_behav"]}
        # roles and gold answers we keep separately
        out["target_text"] = [x["target_text"] for x in batch]
        return out

In [6]:
def ips_reinforce_loss(model, batch, ips_clip: float = 5.0) -> torch.Tensor:
    out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], return_dict=True)
    logprobs = torch.log_softmax(out.logits, dim=-1)
    labels = batch["labels"]
    mask = (labels != -100)

    # sequence log-prob
    labels_safe = torch.where(mask, labels, torch.zeros_like(labels))
    tok_lp = torch.gather(logprobs, -1, labels_safe.unsqueeze(-1)).squeeze(-1)
    seq_logp = (tok_lp * mask).sum(dim=1)

    reward     = batch["reward"].to(seq_logp.dtype)
    logp_behav = batch["logp_behavior"].to(seq_logp.dtype)
    has_behav  = (batch["has_behav"] > 0.5)

    # importance ratio
    with torch.no_grad():
        diff   = seq_logp.detach() - logp_behav
        ratio  = torch.exp(diff)
        if ips_clip is not None and ips_clip > 0:
            ratio = torch.clamp(ratio, max=ips_clip)
        # if no behavior policy, set c=1
        c = torch.where((reward < 0) & has_behav, ratio, torch.ones_like(ratio))

    # final loss
    per_ex = - c * reward * seq_logp
    loss = per_ex.mean()
    return torch.nan_to_num(loss, nan=0.0, neginf=0.0, posinf=0.0)

In [7]:
# -----------------------------------------
# 6) Trainer subclass to use our loss
# -----------------------------------------
class IPSReinforceTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            return_dict=True,
        )
        loss = ips_reinforce_loss(model, inputs)
        return (loss, outputs) if return_outputs else loss

In [8]:
def normalize_guess(s: str) -> str:
    """Map variations of guesses to a canonical form."""
    s = s.strip().lower()
    mapping = {
        "1": "first",
        "first": "first",
        "1.": "first",
        "2": "second",
        "second": "second",
        "2.": "second",
        "3": "third",
        "third": "third",
        "3.": "third"
    }
    return mapping.get(s, s)


def eval_referencegame_outputs(pred_texts, gold_texts):
    scores = []

    for pred, gold in zip(pred_texts, gold_texts):
        pred = pred.strip()
        gold = gold.strip()

        # 1. Strict prefix check
        if not pred.startswith("Answer:"):
            scores.append(-1)
            continue

        # 2. Extract + normalize guesses
        guess = normalize_guess(pred[len("Answer:"):])
        gold_guess = normalize_guess(
            gold[len("Answer:"):] if gold.startswith("Answer:") else gold
        )

        # 3. Match check
        if guess and guess == gold_guess:
            scores.append(1)
        else:
            scores.append(-1)
    print(f"Scores: {scores}")
    return scores

In [9]:
class AccuracyEarlyStoppingCallback(EarlyStoppingCallback):
    def __init__(self, early_stopping_patience=3, early_stopping_threshold=0.0):
        super().__init__()
        self.early_stopping_patience = early_stopping_patience
        self.early_stopping_threshold = early_stopping_threshold
        self.best_score = None
        self.best_step = None
        self.wait_count = 0
        self.best_model_dir = None

    def on_evaluate(self, args, state, control, metrics, model=None, tokenizer=None, **kwargs):
        current = metrics.get("eval_comprehension_accuracy")
        if current is None:
            return control

        if self.best_score is None or current > self.best_score + self.early_stopping_threshold:
            self.best_score = current
            self.best_step = state.global_step
            self.wait_count = 0

            # Save best model
            self.best_model_dir = os.path.join(args.output_dir, "best_model_llm_v2")
            if os.path.exists(self.best_model_dir):
                shutil.rmtree(self.best_model_dir)
            os.makedirs(self.best_model_dir, exist_ok=True)

            if model is not None:
                model.save_pretrained(self.best_model_dir)
            if tokenizer is not None:
                tokenizer.save_pretrained(self.best_model_dir)

            print(f"🌟 New best model saved at {self.best_model_dir} "
                  f"(step {state.global_step}, epoch {state.epoch:.2f}, acc {current:.4f})")

        else:
            self.wait_count += 1
            if self.wait_count >= self.early_stopping_patience:
                print(f"⏹ Early stopping at step {state.global_step}, "
                      f"best acc was {self.best_score:.4f} at step {self.best_step}")
                control.should_training_stop = True
                control.should_save = True

        return control

    def on_train_end(self, args, state, control, model=None, tokenizer=None, **kwargs):
        if self.best_model_dir and os.path.exists(self.best_model_dir):
            # Copy best model into main output dir
            print(f"✅ Training finished. Copying best model from {self.best_model_dir} → {args.output_dir}")
            for item in os.listdir(self.best_model_dir):
                s = os.path.join(self.best_model_dir, item)
                d = os.path.join(args.output_dir, item)
                if os.path.isdir(s):
                    if os.path.exists(d):
                        shutil.rmtree(d)
                    shutil.copytree(s, d)
                else:
                    shutil.copy2(s, d)

            print(f"📌 Best accuracy: {self.best_score:.4f} (step {self.best_step})")

        return control


In [10]:
def load_tokenizer(model_ckpt: str):
    tok = AutoTokenizer.from_pretrained(model_ckpt, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok

def _bf16_supported() -> bool:
    if not torch.cuda.is_available():
        return False
    major, minor = torch.cuda.get_device_capability()
    return major >= 8  # Ampere (8.0) or newer

def load_qlora_base(model_ckpt: str):
    compute_dtype = torch.bfloat16 if _bf16_supported() else torch.float16

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    
    if not torch.cuda.is_available():
        raise RuntimeError("QLoRA 4-bit requires CUDA; no GPU visible.")
    device_index = torch.cuda.current_device()  # usually 0 on your A100 box
    device_map = {"": 0}
    
    model = AutoModelForCausalLM.from_pretrained(
        model_ckpt,
        quantization_config=bnb_config,
        device_map=device_map,
        attn_implementation="eager",  # more stable while debugging
    )
    model.config.use_cache = False
    try:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    except TypeError:
        model.gradient_checkpointing_enable()

    # Prepare for k-bit training (fix layer norms, input grads, etc.)
    from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
    try:
        model = prepare_model_for_kbit_training(
            model, use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
    except TypeError:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

    #"""
    peft_config = LoraConfig(
        r=16,
        lora_alpha=8,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj"],  # only these, per paper
    )
    
    model = get_peft_model(model, peft_config)
    return model
    #"""

def main():
    # --- 0) Set seed
    random.seed(9)
    
    # --- 1) Split data ---
    comp_rows = [r for r in data if r["role"] == "comp"]
    gen_rows  = [r for r in data if r["role"] == "gen"]

    train_comp, dev_comp = train_test_split(
        comp_rows, test_size=0.15, random_state=42, shuffle=True)

    train_rows = gen_rows + train_comp
    dev_rows   = dev_comp   # dev contains only comprehension
    
    print(len(train_rows))
    print(len(dev_rows))

    # --- 2) Tokenizer & QLoRA model ---
    base_model_name = "imge/rf_llama_merged_llms_v1"
    tokenizer = load_tokenizer(base_model_name)
    tokenizer.padding_side = "left"

    # first load the base model
    model = load_qlora_base(base_model_name)

    # --- 3) Dataset/Collator ---
    train_dataset = BanditDataset(train_rows, tokenizer, include_answer=True)
    global dev_dataset
    dev_dataset   = BanditDataset(dev_rows, tokenizer, include_answer=False)
    data_collator = PadCollator(
        pad_id=tokenizer.pad_token_id,
        padding_side=tokenizer.padding_side
    )

    # --- 4) Trainer args ---
    use_bf16 = _bf16_supported()
    args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=32,
        gradient_accumulation_steps=1,
        learning_rate=1e-4,
        num_train_epochs=15,
        lr_scheduler_type="constant",
        warmup_ratio=0,
        weight_decay=0.1,
        logging_steps=1,
        save_strategy="epoch",
        save_total_limit=2,
        evaluation_strategy="epoch",
        load_best_model_at_end=True,
        run_name="whyareurunnin_llms_v2",
        bf16=use_bf16,
        fp16=not use_bf16,
        optim="paged_adamw_8bit",
        remove_unused_columns=False,
    )

    # --- 5) Custom compute_metrics using generation ---

    def compute_metrics(eval_pred):
        print("Compute Acc Metric")
        gen_texts = []
        gold_texts = [ex["target_text"] for ex in dev_dataset]

        eval_dataloader = torch.utils.data.DataLoader(
            dev_dataset,
            batch_size=1, #args.per_device_eval_batch_size,
            collate_fn=data_collator,
        )

        model.eval()
        for batch in eval_dataloader:
            input_ids = batch["input_ids"].to(model.device)
            attention_mask = batch["attention_mask"].to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=32,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.pad_token_id,
                    do_sample=False,
                )
                
            gen_out = outputs[:, input_ids.shape[1]:].cpu()
            texts = tokenizer.batch_decode(gen_out, skip_special_tokens=True)
            gen_texts.extend(texts)
            
            torch.cuda.empty_cache()

        # --- log a few samples ---
        save_dir = "eval_samples_llms_v1_2"
        os.makedirs(save_dir, exist_ok=True)
        step_file = os.path.join(save_dir, f"epoch_{trainer.state.epoch:.0f}.json")

        with open(step_file, "w") as f:
            json.dump(
                [{"gold": g, "pred": p} for g, p in zip(gold_texts[:10], gen_texts[:10])],
                f,
                indent=2,
                ensure_ascii=False
            )

        # normal accuracy calc
        scores = eval_referencegame_outputs(gen_texts, gold_texts)
        acc = sum(1 for s in scores if s == 1) / len(scores) if scores else 0.0

        print(f"Acc: {acc}")
        return {"comprehension_accuracy": acc}


    # --- 6) Trainer ---
    trainer = IPSReinforceTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=dev_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[AccuracyEarlyStoppingCallback(early_stopping_patience=3)],
    )


    print("Start training")
    trainer.train()

    print("Save the model")
    # saves PEFT adapter weights
    trainer.save_model()          

    # ensure correct padding side for decoder-only models
    tokenizer.padding_side = "left"

    tokenizer.save_pretrained(OUTPUT_DIR)



    # push adapters
    model.push_to_hub("imge/reinforce_llama_adapters_llms_v2")
    tokenizer.push_to_hub("imge/reinforce_llama_adapters_llms_v2")
    
    print("FINISHED :D")

In [11]:
main()

2070
168


Loading checkpoint shards: 100%|██████████| 7/7 [00:05<00:00,  1.40it/s]
/project/train_env/lib/python3.10/site-packages/transformers/training_args.py:1559: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


Start training


wandb: Currently logged in as: yuezuencue to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Comprehension Accuracy
1,42.855200,0.000000,0.000000
2,63.201800,0.000000,0.000000
3,107.895700,0.000000,0.000000
4,52.415900,0.000000,0.000000


...

Scores: [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1]
Acc: 0.0
Compute Acc Metric


/project/train_env/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/project/train_env/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


Scores: [-1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1]
Acc: 0.0
⏹ Early stopping at step 260, best acc was 0.0000 at step 65
✅ Training finished. Copying best model from /project/testing_finetuning/best_model_llm_v2 → /project/testing_finetuning
📌 Best accuracy: 0.0000 (step 65)
Save the model


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            
New Data Upload                         : |          |  0.00B /  0.00B            

  ...p679_ol2g/adapter_model.safetensors:   1%|▏         |  555kB / 37.8MB            

Processing Files (0 / 1)                :   1%|▏         |  555kB / 37.8MB,  462kB/s  
New Data Upload                         :   1%|▏         |  555kB / 37.8MB,  462kB/s  

  ...p679_ol2g/adapter_model.safetensors:   1%|▏         |  555kB / 37.8MB            

Processing Files (0 / 1)                :   3%|▎         | 1.11MB / 37.8MB,  694kB/s  
New Data Upload                         :   3%|▎         | 1.11MB / 37.8MB,  694kB/s  

Processing Files (0 / 1)                :  10%|█         | 3.89MB / 37.8MB, 2.16MB/s  
New Data Upload                         :  10%|█         | 3.89MB / 37.8MB, 2.16MB/s  

Processing Files (0 / 1)                :  21%|██        | 7.77MB / 37.8MB, 3.88MB/s  
New Data Upload                         :  21

FINISHED :D


### ACC CHANGES

Training v2 finally resulted in some first acc values that are NOT 0:

Epoch 	Training Loss 	Validation Loss 	Comprehension Accuracy
1 	-167.809500 	0.000000 	0.021505
2 	3.225400 	0.000000 	0.000000
3 	91.755200 	0.000000 	0.000000
4 	17.029600 	0.000000 	0.000000



Have to reduce batch size to 16, and gradient accumulation increase to 2 bc otherwise OOM

In [12]:
! whoami

iyuzuncuoglu


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Tried out the losses without IPS coefficient, with the normal loss, and these were the results.

Step 	Training Loss
1 	518.321400
2 	406.030500
3 	-916.910800
4 	-245.761400
5 	-996.767900
6 	-439.983500
7 	27.368200
8 	-303.980600
9 	98.382900
10 	1272.444300
11 	581.502000
12 	-508.267100
13 	-1107.622900
14 	-2049.277300
15 	715.464200
16 	97.236000
17 	-1123.329300
18 	734.981900
19 	-921.665000
20 	2865.062700
21 	109.311600
22 	-711.254700
23 	-748.666000
24 	-454.268800
25 	-2073.575900
26 	-1976.982200
27 	-246.352200
28 	-2102.958000
29 	-2487.609900
30 	704.824800
31 	-836.676900
32 	-1319.440900
33 	-2682.269300


there is MOST DEFENITELY an issue with my loss somewher and I need to calmly debug it ... UFFF T_T - there was no issue. The issue was the IPS lol


Batch Size: 32 and data around 857]

Steps per epoch: 857/32 = ~27 Steps per epoch

Epoch 1 = steps 1–27

Epoch 2 = steps 28–54

Epoch 3 = steps 55–81

Epoch 4 = steps 82–108

So for the first model, end of epoch 1 has been chosen as the best model :)

In [13]:
from transformers import AutoModel

model = AutoModel.from_pretrained("imge/reinforce_llama_v1")
model.push_to_hub("imge/reinforce_llama_v1")

OSError: imge/reinforce_llama_v1 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

In [ ]:
!printenv